# How the energy grid resolution affects the xs perturbation

In [ ]:
import sandy
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

## Retrieve ENDF-6 file

In [ ]:
tape = sandy.get_endf6_file("endfb_80", "xs", 942390)

## Create PENDF files by parametrically varying reconstruction tolerance ERR

In [ ]:
pendf_1 = tape.get_pendf(
    purr=False,
    heatr=False,
    gaspr=False,
    verbose=True,
    err=1,
)
xs_1 = sandy.Xs.from_endf6(pendf_1)

In [ ]:
pendf_01 = tape.get_pendf(err=0.1)
xs_01 = sandy.Xs.from_endf6(pendf_01)

In [ ]:
pendf_001 = tape.get_pendf(err=0.01)
xs_001 = sandy.Xs.from_endf6(pendf_001)

In [ ]:
pendf_0001 = tape.get_pendf(err=0.001)
xs_0001 = sandy.Xs.from_endf6(pendf_0001)

## Extract cross sections and perturb them in energy bin

In [ ]:
pert_coeff = 1
ipert = 25

egrid = sandy.energy_grids.SCALE238
estart = egrid[ipert]
estop = egrid[ipert+1]

pert = sandy.Pert([1, 1 + pert_coeff], index=[estart, estop])

print(
    f"""
Perturb fission xs in range ({estart}, {estop}) eV by a factor {pert_coeff*100} %.
Perturbation object is:
{pert}
"""
 )

In [ ]:
mat = tape.mat[0]
mt = 18

xspert_1 = xs_1.custom_perturbation(mat, mt, pert)
xspert_01 = xs_01.custom_perturbation(mat, mt, pert)
xspert_001 = xs_001.custom_perturbation(mat, mt, pert)
xspert_0001 = xs_0001.custom_perturbation(mat, mt, pert)

## Analyize how the perturbation was implemented

In [ ]:
def apply_mask(xs, estart, estop):
    mask = (xs.data.index >= estart) & (xs.data.index <= estop)
    return xs.data.loc[mask]

### ERR=1

In [ ]:
print(
    f"""
No point was was found in the selected energy interval with a very low reconstruction tolerance.

{apply_mask(xs_1, estart, estop)[mat, mt].rename("ORIGINAL")}

Points {estart} and {estop} were interpolated and then perurbed in the perturbed xs.

{apply_mask(xspert_1, estart, estop)[mat, mt].rename("PERTURBED")}
"""
)

### ERR=0.001

In [ ]:
print(
    f"""
Many points were found.

{apply_mask(xs_0001, estart, estop)[mat, mt].rename("ORIGINAL")}

Point {estop} was added to the perturbed xs because not present in the original file.

{apply_mask(xspert_0001, estart, estop)[mat, mt].rename("PERTURBED")}
"""
)

### Visual analysis

In [ ]:
fig, ax = plt.subplots()

count = ((xspert_0001.data.index >= estart) & (xspert_0001.data.index <= estop)).sum()
ax = xspert_0001.data[mat, mt].plot(marker="s", ax=ax, ls="--", lw=.5, label=f"ERR=0.001 - {count} perturbed points")

count = ((xspert_001.data.index >= estart) & (xspert_001.data.index <= estop)).sum()
ax = xspert_001.data[mat, mt].plot(marker="s", ax=ax, ls="--", lw=.5, label=f"ERR=0.01 - {count} perturbed points")

count = ((xspert_01.data.index >= estart) & (xspert_01.data.index <= estop)).sum()
ax = xspert_01.data[mat, mt].plot(marker="d", ax=ax, ls="--", lw=.5, label=f"ERR=0.1 - {count} perturbed points")

count = ((xspert_1.data.index >= estart) & (xspert_1.data.index <= estop)).sum()
ax = xspert_1.data[mat, mt].plot(marker="x", ax=ax, ls="--", lw=.5, label=f"ERR=1.0 - {count} perturbed points")

ax = xs_0001.data[mat, mt].plot(ls="--", lw=.5, c="k", ax=ax)

ax.set(xlim=(0.15, 0.25), ylim=(300, 2000), xlabel='energy / $eV$', ylabel="fission xs / $b$")
ax.legend(loc=4)

fig.tight_layout()

## Conclusion

Using a low value for the reconstruction tolerance parameter ERR is necessary to reproduce the expected perturbation behavior.

Alternatively, additional points must be added manually to the cross sections.